# Laboratorul 9 -> Procesarea textelor cu ajutorul LLMs

In Melk, in urma unor inundatii produse de Dunare, o parte din versurile unor poezii (aflate in volumele de poezii din biblioteca) s-au degradat. Folositi un LLM pentru a completa versurile lipsa (avand in vedere ca primul vers din fiecare strofa s-a pastrat intact).

Rezolvarea abordata: folositi un LLM pre-antrenat si adaptat la un corpus de poezii si analizati influenta parametrilor (inclusiv a tokenizer-ului) asupra calitatii textului generat. Adaptarea se realizeaza printr-un proces de fine-tuning pe un corpus de poezii (deci pipeline-ul de optimizare trebuie dezvoltat si implementat de catre student).

In [2]:
import numpy as np
import prelucrarea_datelor
from prelucrarea_datelor import extrage_strofe

### Prelucrarea datelor

In [3]:
import random
import math

cale_folder = "poetry"

poezii = extrage_strofe(cale_folder)
print(len(poezii))

# trebuie impartit setul de date in train si test
# setul de date pentru train va fi modificat a.i. sa fie similar cu un prompt pe care sa-l recunoasca LLM-ul

random.seed(5)
indexes = [i for i in range(len(poezii))]

indexTrain = np.random.choice(indexes, int(0.8 * len(poezii)), replace = False)
indexTest = [i for i in indexes if i not in indexTrain]

trainSet = [poezii[i] for i in indexTrain]
testSet = [poezii[i] for i in indexTest]

429


Prelucrarea corpusului

In [4]:
# se va folosi aceeasi structura de prompt/query din problema rezolvata cu LLM-ul neantrenat

def create_prompt(primele_versuri,poezia_intreaga):
    message = ""
    for i, vers in enumerate(primele_versuri):
        message += f"Stanza {i+1}, Line 1: {vers}\n[...rest of the stanza is missing...]\n\n"

    # system -> defineste "personalitatea"
    # user -> defineste cerinta efectiva
    messages = [
    {"role": "system", "content": "You are an expert in poetry. Restore the missing lines of the provided stanzas. Do not add new stanzas. Provide only the completed text of the poem. Keep in mind that the stanzas have to make sense and rhyme."},
    {"role": "user", "content": f"Complete these stanzas knowing only the first lines:\n\n{message}"},
    {"role":"assistant", "content": poezia_intreaga}
    ]

    return messages

def transform_corpus(corpus):
    # corpusul contine mai multe poezii -> va trebui sa le luam pe fiecare in parte si sa adaugam prompt-ul corespunzator
    trainSample = []

    for poezie in corpus:
        primele_versuri = [strofa['primul_vers'] for strofa in poezie['strofe']]

        # va fi nevoie de ea pentru a ajuta LLM-ul sa se antreneze
        poezie_reconstruita = ""
        for i, strofa in enumerate(poezie['strofe']):
            poezie_reconstruita += f"Stanza {i+1}:\n{strofa['primul_vers']}\n{"\n".join(strofa['restul'])}\n\n"

        message = create_prompt(primele_versuri,poezie_reconstruita)
        trainSample.append(message)

    return trainSample

# NU SUNT INCA TOKENIZATE -> doar au primit structura de prompt
trainSample = transform_corpus(trainSet)
testSample = transform_corpus(testSet)

### Se comprima modelul pentru a rula mai repede antrenarea propriu-zisa

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# reconfigurarea -> model mai mic, organizat pe 4 biti in loc de 16 (cum era initial)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# incarcarea modelului
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

C:\Facultate\Semestrul4\AI\Lab9\.venv1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 291.05it/s]


### Antrenarea

In [6]:
from PoetryDataset import PoetryDataset
from torch.utils.data import DataLoader
import torch

# decide unde ar fi mai bine sa ruleze modelul (unde are disponibilitate)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Folosesc device-ul: {device}")

# pregatirea setului de date de antrenare
train_dataset = PoetryDataset(trainSample, tokenizer, max_length=512)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)


optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

# seteaza modul de functionare al modelului (nu da train propriu-zis)
model.train()

# bucla de antrenare propriu-zisa
for epoch in range(3):
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss mediu: {total_loss / len(train_loader):.4f}")

Folosesc device-ul: cuda
Epoch 1 | Loss mediu: 1.4313
Epoch 2 | Loss mediu: 1.1056
Epoch 3 | Loss mediu: 0.9757


### Testarea pe o poezie din setul de test

In [19]:
# s-a ales o poezie "random" pentru testare
poezie_test = testSample[21]

# se pregateste input-ul tokenizat
input_text = tokenizer.apply_chat_template(
    poezie_test[:2], # se trimit doar mesajele de system si aser, fara assistant (care e unul dintre raspunsurile posibile)
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(input_text, return_tensors="pt").to(device)

# se seteaza modul de functionare al modelului la eval
# foloseste toate conexiunile (spre deosebire de modul train care dezactiveaza neuroni, pt a preveni overfitting)
model.eval()

# generarea output-ului
output = model.generate(
    **inputs,
    max_new_tokens=600,
    temperature=0.8,
    do_sample=True,
    repetition_penalty=1.1
)

# decoding -> pentru a primi un raspuns in limbaj natural
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
doar_strofele = tokenizer.decode(output[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)

print(generated_text)

system
You are an expert in poetry. Restore the missing lines of the provided stanzas. Do not add new stanzas. Provide only the completed text of the poem. Keep in mind that the stanzas have to make sense and rhyme.
user
Complete these stanzas knowing only the first lines:

Stanza 1, Line 1: When Peter who was a country jake
[...rest of the stanza is missing...]

Stanza 2, Line 1: And after service when his Ma
[...rest of the stanza is missing...]


assistant
Stanza 1:
When Peter who was a country jake
Saw the moon shining so bright
He said "Oh my sweet Mother dear
What time it be before you get home?"

Stanza 2:
And after service when his Ma
Asked him how he had spent his day
He replied with a sigh of despair
I could tell she felt very sad
Because she knew I would return soon


### BLEU - Bilangual Evaluation Understudy

In [21]:
import nltk
from nltk.translate.bleu_score import corpus_bleu
from nltk.tokenize import word_tokenize

poezie_test = testSet[21]
strof = poezie_test['strofe']

predicted = []
for strofa in strof:
    s = ""
    s+= strofa['primul_vers']
    for vers in strofa['restul']:
        s+=vers

    cuv = word_tokenize(s)
    predicted.append([cuv])

print(predicted)
print()

generated = []
for strofa_gen in doar_strofele.split("\n\n"):
    cuv = word_tokenize(strofa_gen)
    generated.append(cuv)

print(generated)


# se echilibreaza nr de strofe
diferenta = len(predicted) - len(generated)

if diferenta > 0:
    for _ in range(diferenta):
        generated.append([])
elif diferenta < 0:
    for _ in range(-diferenta):
        predicted.append([[]])

score = corpus_bleu(predicted, generated)
print()
print(score)

[[['When', 'Peter', 'who', 'was', 'a', 'country', 'jakeA', 'visit', 'to', 'a', 'church', 'did', 'makeHe', 'sat', 'with', 'pleased', 'look', 'on', 'his', 'faceAs', 'if', 'indeed', 'in', 'Heaven', "'s", 'place', '.']], [['And', 'after', 'service', 'when', 'his', 'MaPraised', 'him', 'aloud', 'to', 'his', 'kind', 'PaHe', 'said', ',', '“', 'Of', 'course', 'I', 'sat', 'quite', 'stillAnd', 'watched', 'the', 'preacher', "'s", 'wives', 'so', 'illAll', 'dressed', 'in', 'nighties', ',', 'though', 'their', 'hairWas', 'primped', 'and', 'curled', 'as', 'for', 'a', 'fair', '.', '”']]]

[['Stanza', '1', ':', 'When', 'Peter', 'who', 'was', 'a', 'country', 'jake', 'Saw', 'the', 'moon', 'shining', 'so', 'bright', 'He', 'said', '``', 'Oh', 'my', 'sweet', 'Mother', 'dear', 'What', 'time', 'it', 'be', 'before', 'you', 'get', 'home', '?', "''"], ['Stanza', '2', ':', 'And', 'after', 'service', 'when', 'his', 'Ma', 'Asked', 'him', 'how', 'he', 'had', 'spent', 'his', 'day', 'He', 'replied', 'with', 'a', 'sigh',